# Projet 7 — VLMs légers pour légender ou juger des images

**Objectif** : tester deux petits modèles vision-langage (VLM) sur le même jeu d'images (Flickr8k), comparer leurs versions **16 bits, 8 bits et 4 bits**, puis :

- **Option A (principale)** : générer des légendes, les évaluer face aux légendes humaines et préparer un jeu de données prêt pour entraîner une **LoRA** ;
- **Bonus option B** : utiliser le VLM comme **juge** (note de 1 à 5) et mesurer sa **corrélation** avec un jugement humain ;
- **Démo finale** : interface Gradio (charger une image → légende ou score, avec choix du niveau de quantification).

| Modèle | Identifiant Hugging Face | Taille |
|---|---|---|
| Qwen2-VL-2B | `Qwen/Qwen2-VL-2B-Instruct` | ~2,2 B paramètres |
| SmolVLM | `HuggingFaceTB/SmolVLM-Instruct` | ~2,2 B paramètres |

> ⚙️ **Avant de commencer (Colab)** : *Exécution → Modifier le type d'exécution → GPU (T4)*. Exécute ensuite les cellules dans l'ordre.
> Les résultats sont sauvegardés au fur et à mesure : si Colab se déconnecte, relance simplement les cellules, les configurations déjà calculées sont reprises.

**Plan**
1. Installation et configuration
2. Préparation des données (Flickr8k)
3. Chargement des VLMs (16 / 8 / 4 bits)
4. Génération des légendes + mesures (mémoire, vitesse)
5. Évaluation (BLEU, ROUGE-L, CIDEr, CLIPScore, fidélité au 16 bits)
6. Option A : export du jeu de données pour LoRA (+ entraînement optionnel)
7. Bonus option B : le VLM comme juge, corrélation avec l'humain
8. Démo Gradio
9. Conclusion

## 1. Installation et configuration

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate bitsandbytes datasets pycocoevalcap gradio num2words scipy

In [ ]:
import os, gc, json, re, time, random, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

assert torch.cuda.is_available(), "Aucun GPU détecté : active un GPU (Exécution → Modifier le type d'exécution)."
print("GPU :", torch.cuda.get_device_name(0))
print("Mémoire GPU totale : %.1f Go" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

# bfloat16 seulement sur GPU Ampere ou plus récent (A100, L4...). Sur T4 → float16.
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("Type des poids en 16 bits :", DTYPE)

# ------------------------- CONFIGURATION -------------------------
SEED = 42
N_IMAGES = 300            # taille du sous-ensemble évalué (le sujet conseille 500 à 2000 ; 300 ≈ 1 h sur T4)
MAX_SIDE = 512            # redimensionnement des images (côté le plus long, en pixels)
MAX_NEW_TOKENS = 40       # longueur max d'une légende

MODELS = {
    "Qwen2-VL-2B": "Qwen/Qwen2-VL-2B-Instruct",
    "SmolVLM-2B":  "HuggingFaceTB/SmolVLM-Instruct",
}
PRECISIONS = ["16bit", "8bit", "4bit"]

CAPTION_PROMPT = "Describe this image in one short sentence."

# Option : sauvegarder sur Google Drive pour ne rien perdre en cas de déconnexion
USE_DRIVE = True   # dossier partagé de l'équipe : MyDrive/projet7_vlm
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/projet7_vlm")
else:
    ROOT = Path("/content/projet7_vlm") if Path("/content").exists() else Path("projet7_vlm")

DATA_DIR = ROOT / "data"
IMG_DIR = DATA_DIR / "images"
RES_DIR = ROOT / "results"
for d in (IMG_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Dossier de travail :", ROOT)

## 2. Préparation des données — Flickr8k

Flickr8k contient ~8 000 images, chacune avec **5 légendes humaines**. On prend un sous-ensemble aléatoire (graine fixe) de la partie *test*, on **redimensionne** les images (côté max = `MAX_SIDE`) et on les enregistre sur disque : **les mêmes images** serviront à toutes les configurations (16 / 8 / 4 bits), ce qui rend la comparaison équitable.

In [ ]:
from datasets import load_dataset

def prepare_image(img, max_side=MAX_SIDE):
    # Convertit en RGB et réduit l'image pour que son plus grand côté fasse max_side pixels.
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side), Image.BICUBIC)
    return img

META_PATH = DATA_DIR / "subset.json"

if META_PATH.exists():
    subset = json.loads(META_PATH.read_text())
    print(f"Sous-ensemble déjà préparé : {len(subset)} images")
else:
    ds = load_dataset("jxie/flickr8k", split="test")          # 1000 images, colonnes image + caption_0..4
    caption_cols = sorted(c for c in ds.column_names if c.startswith("caption"))
    idx = sorted(random.Random(SEED).sample(range(len(ds)), min(N_IMAGES, len(ds))))
    subset = []
    for i in tqdm(idx, desc="Préparation des images"):
        ex = ds[i]
        img_id = f"img_{i:04d}"
        prepare_image(ex["image"]).save(IMG_DIR / f"{img_id}.jpg", quality=95)
        subset.append({"id": img_id, "refs": [ex[c].strip() for c in caption_cols]})
    META_PATH.write_text(json.dumps(subset, indent=1, ensure_ascii=False))
    print(f"{len(subset)} images préparées dans {IMG_DIR}")

REFS = {s["id"]: s["refs"] for s in subset}
IDS = [s["id"] for s in subset]

def load_img(img_id):
    return Image.open(IMG_DIR / f"{img_id}.jpg").convert("RGB")

In [ ]:
# Aperçu de quelques images et de leurs légendes humaines
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, img_id in zip(axes, IDS[:4]):
    ax.imshow(load_img(img_id)); ax.axis("off")
    ax.set_title("\n".join(r[:45] for r in REFS[img_id][:2]), fontsize=8)
plt.tight_layout(); plt.show()

## 3. Chargement des VLMs en 16, 8 et 4 bits

La **quantification** stocke les poids du modèle avec moins de bits :

| Précision | Méthode | Mémoire des poids (≈ pour 2 B paramètres) |
|---|---|---|
| 16 bits | float16 / bfloat16 | ~4,4 Go |
| 8 bits | `bitsandbytes` LLM.int8 | ~2,5 Go |
| 4 bits | `bitsandbytes` NF4 + double quantification | ~1,5 Go |

On ne ré-entraîne pas les VLMs : on fait uniquement de **l'inférence**, comme le demande le sujet.

In [ ]:
import transformers
from packaging import version
from transformers import AutoProcessor, BitsAndBytesConfig
# Nouvelles versions de transformers : argument `dtype` ; anciennes : `torch_dtype`
DTYPE_KW = {"dtype": DTYPE} if version.parse(transformers.__version__) >= version.parse("4.56") else {"torch_dtype": DTYPE}
print("transformers", transformers.__version__)
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM

def quant_config(precision):
    if precision == "16bit":
        return None
    if precision == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    if precision == "4bit":
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    raise ValueError(precision)

def load_vlm(name, precision):
    model_id = MODELS[name]
    # On limite la résolution vue par le modèle pour garder un nombre de tokens image raisonnable
    if "Qwen" in model_id:
        proc_kwargs = dict(min_pixels=128 * 28 * 28, max_pixels=512 * 28 * 28)
    elif "SmolVLM" in model_id:
        proc_kwargs = dict(size={"longest_edge": 2 * 384})
    else:
        proc_kwargs = {}
    try:
        processor = AutoProcessor.from_pretrained(model_id, **proc_kwargs)
    except Exception:
        processor = AutoProcessor.from_pretrained(model_id)
    model = AutoVLM.from_pretrained(model_id, device_map="cuda:0",
                                    quantization_config=quant_config(precision), **DTYPE_KW)
    model.eval()
    return model, processor

def unload(*objs):
    for o in objs:
        del o
    gc.collect(); torch.cuda.empty_cache()

@torch.inference_mode()
def vlm_generate(model, processor, image, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=text, images=[image], return_tensors="pt")
    inputs = {k: (v.to(model.device, DTYPE) if torch.is_floating_point(v) else v.to(model.device))
              for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

def clean_caption(text):
    # Garde la première ligne / phrase et retire un éventuel préfixe « Assistant: ».
    text = re.sub(r"^\s*(assistant\s*:)\s*", "", text.strip(), flags=re.I)
    text = text.split("\n")[0].strip()
    m = re.match(r"(.{15,}?[.!?])(\s|$)", text)
    return (m.group(1) if m else text).strip()

In [ ]:
# Test rapide : un modèle en 4 bits sur une image
m, p = load_vlm("SmolVLM-2B", "4bit")
img = load_img(IDS[0])
print("VLM  :", clean_caption(vlm_generate(m, p, img, CAPTION_PROMPT)))
print("Réf. :", REFS[IDS[0]][0])
unload(m, p); del m, p; gc.collect(); torch.cuda.empty_cache()

## 4. Génération des légendes et mesures

Pour chaque modèle × précision, on mesure :
- **mémoire des poids** (`get_memory_footprint`) et **pic de mémoire GPU** pendant la génération ;
- **temps de chargement** et **temps moyen par image** ;
- les **légendes** générées (sauvegardées en CSV).

Les configurations déjà calculées sont rechargées depuis `results/` (reprise après déconnexion).

In [ ]:
def run_config(name, precision, ids=IDS):
    tag = f"{name}__{precision}"
    cap_path, stat_path = RES_DIR / f"captions_{tag}.csv", RES_DIR / f"stats_{tag}.json"
    if cap_path.exists() and stat_path.exists():
        df = pd.read_csv(cap_path)
        if set(df["id"]) >= set(ids):
            print(f"[déjà calculé] {tag}")
            return
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model, processor = load_vlm(name, precision)
    load_time = time.time() - t0
    footprint = model.get_memory_footprint() / 1e9

    vlm_generate(model, processor, load_img(ids[0]), CAPTION_PROMPT)   # échauffement (non chronométré)
    torch.cuda.reset_peak_memory_stats()

    rows, times = [], []
    for img_id in tqdm(ids, desc=tag):
        img = load_img(img_id)
        torch.cuda.synchronize(); t = time.time()
        raw = vlm_generate(model, processor, img, CAPTION_PROMPT)
        torch.cuda.synchronize(); times.append(time.time() - t)
        rows.append({"id": img_id, "raw": raw, "caption": clean_caption(raw)})

    stats = {"model": name, "precision": precision, "load_time_s": load_time,
             "weights_GB": footprint, "peak_GPU_GB": torch.cuda.max_memory_allocated() / 1e9,
             "time_per_image_s": float(np.mean(times)), "n_images": len(ids)}
    pd.DataFrame(rows).to_csv(cap_path, index=False)
    stat_path.write_text(json.dumps(stats, indent=1))
    print(stats)
    unload(model, processor); del model, processor
    gc.collect(); torch.cuda.empty_cache()

for name in MODELS:
    for prec in PRECISIONS:
        run_config(name, prec)

## 5. Évaluation

**Métriques par rapport aux 5 légendes humaines** :
- **BLEU-1 / BLEU-4** : recouvrement de mots et de groupes de 4 mots ;
- **ROUGE-L** : plus longue sous-séquence commune ;
- **CIDEr** : la métrique de référence en légendage (pondère les mots informatifs) ;
- **CLIPScore** : accord image ↔ légende mesuré par CLIP, **sans** référence humaine.

**Fidélité au 16 bits** : à quel point la version quantifiée produit la même légende que la version 16 bits (BLEU-4 contre la légende 16 bits, % de légendes identiques).

> Remarque : les VLMs modernes écrivent des phrases plus longues et détaillées que les légendes de Flickr8k, donc BLEU/CIDEr sont bas en absolu. Ce qui compte ici, c'est la **comparaison** entre modèles et précisions. Le CLIPScore complète en évaluant la justesse visuelle.

In [ ]:
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

def tok(s):
    t = " ".join(re.findall(r"[a-z0-9]+", str(s).lower()))
    return t if t else "empty"

def text_metrics(preds, refs):
    # preds : {id: légende}, refs : {id: [légendes de référence]}
    gts = {k: [tok(r) for r in refs[k]] for k in preds}
    res = {k: [tok(preds[k])] for k in preds}
    bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    return {"BLEU-1": bleu[0], "BLEU-4": bleu[3], "ROUGE-L": rouge, "CIDEr": cider}

In [ ]:
# CLIPScore (Hessel et al., 2021) : 2.5 * max(cos(image, texte), 0)
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").half().cuda().eval()

def _as_tensor(e):
    # Selon la version de transformers, get_*_features renvoie un tenseur ou un objet de sortie
    if torch.is_tensor(e):
        return e
    for attr in ("image_embeds", "text_embeds", "pooler_output"):
        if getattr(e, attr, None) is not None:
            return getattr(e, attr)
    raise TypeError(type(e))
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.inference_mode()
def clip_image_emb(ids, bs=32):
    embs = []
    for i in range(0, len(ids), bs):
        imgs = [load_img(x) for x in ids[i:i + bs]]
        px = clip_proc(images=imgs, return_tensors="pt")["pixel_values"].cuda().half()
        e = clip_model.get_image_features(pixel_values=px)
        embs.append(torch.nn.functional.normalize(_as_tensor(e).float(), dim=-1).cpu())
    return dict(zip(ids, torch.cat(embs)))

@torch.inference_mode()
def clip_text_emb(texts, bs=64):
    embs = []
    for i in range(0, len(texts), bs):
        t = clip_proc(text=list(texts[i:i + bs]), return_tensors="pt", padding=True,
                      truncation=True, max_length=77)
        e = clip_model.get_text_features(input_ids=t["input_ids"].cuda(), attention_mask=t["attention_mask"].cuda())
        embs.append(torch.nn.functional.normalize(_as_tensor(e).float(), dim=-1).cpu())
    return torch.cat(embs)

IMG_EMB = clip_image_emb(IDS)

def clip_scores(ids, captions):
    T = clip_text_emb([str(c) for c in captions])
    I = torch.stack([IMG_EMB[i] for i in ids])
    return (2.5 * (I * T).sum(-1).clamp(min=0)).numpy()

In [ ]:
CAPS, rows = {}, []
for name in MODELS:
    for prec in PRECISIONS:
        tag = f"{name}__{prec}"
        df = pd.read_csv(RES_DIR / f"captions_{tag}.csv").set_index("id").loc[IDS]
        df["caption"] = df["caption"].fillna("").astype(str)
        CAPS[(name, prec)] = df["caption"].to_dict()
        stats = json.loads((RES_DIR / f"stats_{tag}.json").read_text())
        m = text_metrics(CAPS[(name, prec)], REFS)
        m["CLIPScore"] = float(clip_scores(IDS, df["caption"].tolist()).mean())
        m["mots/légende"] = float(df["caption"].str.split().str.len().mean())
        # Fidélité à la version 16 bits du même modèle
        if prec != "16bit":
            base = CAPS[(name, "16bit")]
            fid = text_metrics(CAPS[(name, prec)], {k: [base[k]] for k in IDS})
            m["BLEU-4 vs 16bit"] = fid["BLEU-4"]
            m["% identique au 16bit"] = 100 * np.mean([tok(CAPS[(name, prec)][k]) == tok(base[k]) for k in IDS])
        rows.append({**stats, **m})

# Référence humaine : 1 légende humaine évaluée contre les 4 autres (ordre de grandeur, non strictement comparable)
human_pred = {k: REFS[k][0] for k in IDS}
h = text_metrics(human_pred, {k: REFS[k][1:] for k in IDS})
h["CLIPScore"] = float(clip_scores(IDS, [human_pred[k] for k in IDS]).mean())
h["mots/légende"] = float(np.mean([len(human_pred[k].split()) for k in IDS]))
rows.append({"model": "Humain (réf. 0 vs 4 autres)", "precision": "-", **h})

results = pd.DataFrame(rows)
results.to_csv(RES_DIR / "resultats_finaux.csv", index=False)
cols = ["model", "precision", "weights_GB", "peak_GPU_GB", "time_per_image_s", "BLEU-1", "BLEU-4",
        "ROUGE-L", "CIDEr", "CLIPScore", "mots/légende", "BLEU-4 vs 16bit", "% identique au 16bit"]
results[cols].round(3)

In [ ]:
# Graphiques de comparaison
res = results[results["precision"] != "-"].copy()
models = list(MODELS)
colors = {"16bit": "#2a6fdb", "8bit": "#f2a541", "4bit": "#3bb273"}
metrics = [("weights_GB", "Mémoire des poids (Go)"), ("peak_GPU_GB", "Pic mémoire GPU (Go)"),
           ("time_per_image_s", "Temps par image (s)"), ("CIDEr", "CIDEr (↑)"),
           ("CLIPScore", "CLIPScore (↑)"), ("BLEU-4", "BLEU-4 (↑)")]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
x = np.arange(len(models)); w = 0.26
for ax, (col, title) in zip(axes.ravel(), metrics):
    for j, prec in enumerate(PRECISIONS):
        vals = [res[(res.model == m) & (res.precision == prec)][col].values[0] for m in models]
        bars = ax.bar(x + (j - 1) * w, vals, w, label=prec, color=colors[prec])
        ax.bar_label(bars, fmt="%.2f", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(models); ax.set_title(title)
    ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].legend(title="Précision")
plt.tight_layout(); plt.savefig(RES_DIR / "comparaison.png", dpi=150); plt.show()

# Compromis mémoire / qualité
fig, ax = plt.subplots(figsize=(7, 5))
for m, marker in zip(models, ["o", "s"]):
    for prec in PRECISIONS:
        r = res[(res.model == m) & (res.precision == prec)].iloc[0]
        ax.scatter(r.weights_GB, r.CIDEr, s=120, marker=marker, color=colors[prec])
        ax.annotate(f"{m}\n{prec}", (r.weights_GB, r.CIDEr), fontsize=8, xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("Mémoire des poids (Go)"); ax.set_ylabel("CIDEr")
ax.set_title("Compromis mémoire / qualité"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(RES_DIR / "compromis_memoire_qualite.png", dpi=150); plt.show()

In [ ]:
# Exemples qualitatifs : mêmes images, 6 configurations
for img_id in IDS[:4]:
    plt.figure(figsize=(4, 4)); plt.imshow(load_img(img_id)); plt.axis("off"); plt.show()
    print("Référence humaine :", REFS[img_id][0])
    for (name, prec), caps in CAPS.items():
        print(f"  {name:12s} {prec:6s} → {caps[img_id]}")
    print("-" * 100)

## 6. Option A — Jeu de données de légendes pour entraîner une LoRA

On exporte les images + légendes de la meilleure configuration (ou d'une configuration choisie) au format utilisé par les scripts d'entraînement LoRA :
- `metadata.jsonl` (format `imagefolder` de 🤗 *diffusers*) ;
- un fichier `.txt` par image (format *kohya_ss*).

Un **mot déclencheur** optionnel (`TRIGGER_WORD`) est ajouté devant chaque légende pour pouvoir activer le style de la LoRA.

In [ ]:
# Choix de la configuration : par défaut, la meilleure en CIDEr. Tu peux forcer ex. ("Qwen2-VL-2B", "4bit").
best = res.sort_values("CIDEr", ascending=False).iloc[0]
LORA_CONFIG = (best.model, best.precision)
TRIGGER_WORD = ""          # ex. "flkr style" ; vide = pas de mot déclencheur
print("Configuration utilisée pour la LoRA :", LORA_CONFIG)

LORA_DIR = ROOT / "lora_dataset" / "train"
if LORA_DIR.exists():
    shutil.rmtree(LORA_DIR)
LORA_DIR.mkdir(parents=True)

with open(LORA_DIR / "metadata.jsonl", "w", encoding="utf-8") as f:
    for img_id in IDS:
        cap = CAPS[LORA_CONFIG][img_id]
        cap = f"{TRIGGER_WORD}, {cap}" if TRIGGER_WORD else cap
        shutil.copy(IMG_DIR / f"{img_id}.jpg", LORA_DIR / f"{img_id}.jpg")
        (LORA_DIR / f"{img_id}.txt").write_text(cap, encoding="utf-8")
        f.write(json.dumps({"file_name": f"{img_id}.jpg", "text": cap}, ensure_ascii=False) + "\n")

shutil.make_archive(str(ROOT / "lora_dataset"), "zip", ROOT / "lora_dataset")
print(f"{len(IDS)} paires image/légende exportées → {ROOT / 'lora_dataset.zip'}")
print(open(LORA_DIR / "metadata.jsonl").readlines()[:3])

### (Optionnel) Entraîner une petite LoRA Stable Diffusion 1.5 avec ces légendes

Mets `RUN_LORA_TRAINING = True` pour lancer un **petit** entraînement (≈ 30–45 min sur T4 pour 500 pas). Le sujet n'exige pas forcément cet entraînement : le livrable principal de l'option A est le jeu de légendes ci-dessus.

In [ ]:
RUN_LORA_TRAINING = False

if RUN_LORA_TRAINING:
    import subprocess
    gc.collect(); torch.cuda.empty_cache()
    subprocess.run("git clone --depth 1 https://github.com/huggingface/diffusers /content/diffusers", shell=True, check=False)
    subprocess.run("pip install -q /content/diffusers peft ftfy tensorboard", shell=True, check=True)
    cmd = f'''accelerate launch --mixed_precision=fp16 /content/diffusers/examples/text_to_image/train_text_to_image_lora.py \
      --pretrained_model_name_or_path="stable-diffusion-v1-5/stable-diffusion-v1-5" \
      --train_data_dir="{LORA_DIR}" --caption_column="text" --image_column="image" \
      --resolution=512 --center_crop --random_flip \
      --train_batch_size=1 --gradient_accumulation_steps=4 --gradient_checkpointing \
      --max_train_steps=500 --learning_rate=1e-4 --lr_scheduler="cosine" --lr_warmup_steps=0 \
      --rank=4 --seed={SEED} --output_dir="{ROOT / 'lora_sd15'}"'''
    subprocess.run(cmd, shell=True, check=True)

    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5",
                                                   torch_dtype=torch.float16).to("cuda")
    pipe.load_lora_weights(str(ROOT / "lora_sd15"))
    prompt = (TRIGGER_WORD + ", " if TRIGGER_WORD else "") + "a dog running on the beach"
    image = pipe(prompt, num_inference_steps=30, generator=torch.Generator("cuda").manual_seed(SEED)).images[0]
    image.save(ROOT / "exemple_lora.png"); display(image)
    del pipe; gc.collect(); torch.cuda.empty_cache()

## 7. Bonus option B — Le VLM comme juge

On demande au VLM de **noter de 1 à 5** si une légende décrit bien l'image. Pour chaque image du jeu de jugement, on lui soumet trois légendes :
- `humaine` : une légende Flickr8k (devrait être bien notée) ;
- `vlm` : une légende générée en partie 4 (qualité variable) ;
- `mauvaise` : la légende d'une **autre** image (devrait être mal notée).

On compare ensuite ses notes à un **jugement humain** avec les corrélations de **Spearman**, **Pearson** et **Kendall**, pour chaque niveau de quantification.

**Annotation humaine** : l'outil ci-dessous affiche chaque paire image/légende et tu cliques une note de 1 à 5 (≈ 10 min pour 120 paires). Si tu n'annotes pas, le notebook utilise un **proxy** (humaine = 5, mauvaise = 1, paires `vlm` exclues) — à signaler clairement dans le rapport.

In [ ]:
# Légendes VLM pour les paires « vlm » : on prend celles de Wafae (Qwen2-VL-2B, 16 bits) si elles existent,
# sinon on les génère ici pour les N_JUDGE premières images (≈ 1 min).
N_JUDGE = 40
VLM_CAPTION_CONFIG = ("Qwen2-VL-2B", "16bit")
cap_path = RES_DIR / f"captions_{VLM_CAPTION_CONFIG[0]}__{VLM_CAPTION_CONFIG[1]}.csv"
own_path = RES_DIR / "judge_vlm_captions.csv"
if own_path.exists():
    VLM_CAPS = pd.read_csv(own_path).set_index("id")["caption"].to_dict()
elif cap_path.exists():
    VLM_CAPS = pd.read_csv(cap_path).set_index("id")["caption"].to_dict()
else:
    m, p = load_vlm(*VLM_CAPTION_CONFIG)
    VLM_CAPS = {i: clean_caption(vlm_generate(m, p, load_img(i), CAPTION_PROMPT)) for i in tqdm(IDS[:N_JUDGE])}
    unload(m, p); del m, p; gc.collect(); torch.cuda.empty_cache()
# On fige ces légendes pour que les paires restent identiques d'une exécution à l'autre
pd.DataFrame({"id": IDS[:N_JUDGE], "caption": [VLM_CAPS[i] for i in IDS[:N_JUDGE]]}).to_csv(own_path, index=False)
VLM_CAPS = pd.read_csv(own_path).set_index("id")["caption"].to_dict()
print(len(VLM_CAPS), "légendes VLM prêtes")

rng = random.Random(SEED)
judge_ids = IDS[:N_JUDGE]
pairs = []
for k, img_id in enumerate(judge_ids):
    other = judge_ids[(k + rng.randint(1, N_JUDGE - 1)) % N_JUDGE]
    pairs += [{"image_id": img_id, "type": "humaine", "caption": REFS[img_id][1]},
              {"image_id": img_id, "type": "vlm", "caption": VLM_CAPS[img_id]},
              {"image_id": img_id, "type": "mauvaise", "caption": REFS[other][1]}]
rng.shuffle(pairs)
pairs = pd.DataFrame(pairs)
pairs.insert(0, "pair_id", range(len(pairs)))

ANNOTATRICE = "fatima"   # chaque membre qui annote met SON prénom ici (wafae, meriem, fatima)
HUMAN_PATH = RES_DIR / f"notes_humaines_{ANNOTATRICE}.csv"
if HUMAN_PATH.exists():
    human = pd.read_csv(HUMAN_PATH)
else:
    human = pairs.assign(human_score=np.nan)
    human.to_csv(HUMAN_PATH, index=False)
print(f"{len(pairs)} paires — déjà annotées : {human['human_score'].notna().sum()}")

In [ ]:
# Outil d'annotation (clique une note ; la sauvegarde est automatique)
import io
import ipywidgets as widgets
from IPython.display import display

state = {"i": int(human["human_score"].notna().sum())}
img_w = widgets.Image(format="jpeg", width=380)
cap_w = widgets.HTML()
prog_w = widgets.HTML()
buttons = [widgets.Button(description=str(s), layout=widgets.Layout(width="60px")) for s in range(1, 6)]

def show():
    i = state["i"]
    if i >= len(human):
        prog_w.value = "<b>✅ Annotation terminée.</b>"; cap_w.value = ""; return
    r = human.iloc[i]
    buf = io.BytesIO(); load_img(r.image_id).save(buf, format="JPEG"); img_w.value = buf.getvalue()
    cap_w.value = f"<p style='font-size:16px'><b>Légende :</b> {r.caption}</p><p>1 = complètement faux … 5 = parfait</p>"
    prog_w.value = f"Paire {i + 1} / {len(human)}"

def on_click(b):
    if state["i"] < len(human):
        human.loc[state["i"], "human_score"] = int(b.description)
        human.to_csv(HUMAN_PATH, index=False)
        state["i"] += 1
    show()

for b in buttons:
    b.on_click(on_click)
show()
display(widgets.VBox([prog_w, img_w, cap_w, widgets.HBox(buttons)]))

In [ ]:
JUDGE_PROMPT = ('Caption: "{caption}"\n'
                "On a scale from 1 to 5, how well does this caption describe the image? "
                "1 = completely wrong, 5 = perfect. Answer with a single digit.")

def parse_score(text):
    m = re.search(r"[1-5]", text)
    return int(m.group()) if m else np.nan

JUDGE_MODELS = list(MODELS)     # les deux modèles, en 16 / 8 / 4 bits
for name in JUDGE_MODELS:
    for prec in PRECISIONS:
        path = RES_DIR / f"judge_{name}__{prec}.csv"
        if path.exists():
            print(f"[déjà calculé] juge {name} {prec}"); continue
        model, processor = load_vlm(name, prec)
        scores = []
        for r in tqdm(pairs.itertuples(), total=len(pairs), desc=f"juge {name} {prec}"):
            out = vlm_generate(model, processor, load_img(r.image_id),
                               JUDGE_PROMPT.format(caption=r.caption), max_new_tokens=5)
            scores.append({"pair_id": r.pair_id, "raw": out, "vlm_score": parse_score(out)})
        pd.DataFrame(scores).to_csv(path, index=False)
        unload(model, processor); del model, processor
        gc.collect(); torch.cuda.empty_cache()

In [ ]:
from scipy.stats import spearmanr, pearsonr, kendalltau

# Moyenne des notes de toutes les annotatrices (fichiers notes_humaines_<prénom>.csv du dossier partagé)
_files = sorted(RES_DIR.glob("notes_humaines_*.csv"))
_all = pd.concat([pd.read_csv(f)[["pair_id", "human_score"]].assign(annotatrice=f.stem.split("_")[-1]) for f in _files])
human = pd.read_csv(HUMAN_PATH)
_mean = _all.dropna(subset=["human_score"]).groupby("pair_id")["human_score"].mean().reset_index()
print("Annotatrices :", [f.stem.split("_")[-1] for f in _files])

# Accord entre annotatrices (si au moins deux)
_wide = _all.pivot_table(index="pair_id", columns="annotatrice", values="human_score")
_names = list(_wide.columns)
for a in range(len(_names)):
    for b in range(a + 1, len(_names)):
        _d = _wide[[_names[a], _names[b]]].dropna()
        if len(_d) >= 10:
            print(f"Accord {_names[a]} / {_names[b]} : Spearman = {spearmanr(_d.iloc[:, 0], _d.iloc[:, 1])[0]:.2f} sur {len(_d)} paires")

if len(_mean) >= 20:
    gt = _mean; source = f"moyenne des notes humaines ({len(_files)} annotatrice(s))"
else:
    proxy = human[human["type"] != "vlm"].copy()
    proxy["human_score"] = proxy["type"].map({"humaine": 5, "mauvaise": 1})
    gt = proxy[["pair_id", "human_score"]]; source = "PROXY (humaine=5 / mauvaise=1) — annote pour un vrai résultat"
print("Vérité terrain utilisée :", source, f"({len(gt)} paires)")

pairs["CLIPScore"] = clip_scores(pairs["image_id"].tolist(), pairs["caption"].tolist())
corr_rows = []
for name in JUDGE_MODELS:
    for prec in PRECISIONS:
        j = pd.read_csv(RES_DIR / f"judge_{name}__{prec}.csv")
        d = pairs.merge(j, on="pair_id").merge(gt, on="pair_id").dropna(subset=["vlm_score"])
        corr_rows.append({"juge": f"{name} {prec}", "n": len(d),
                          "réponses valides %": 100 * j["vlm_score"].notna().mean(),
                          "Spearman": spearmanr(d.vlm_score, d.human_score)[0],
                          "Pearson": pearsonr(d.vlm_score, d.human_score)[0],
                          "Kendall": kendalltau(d.vlm_score, d.human_score)[0],
                          "note moy. humaine": d[d.type == "humaine"].vlm_score.mean(),
                          "note moy. vlm": d[d.type == "vlm"].vlm_score.mean(),
                          "note moy. mauvaise": d[d.type == "mauvaise"].vlm_score.mean()})
d = pairs.merge(gt, on="pair_id")
corr_rows.append({"juge": "CLIPScore (référence)", "n": len(d),
                  "Spearman": spearmanr(d.CLIPScore, d.human_score)[0],
                  "Pearson": pearsonr(d.CLIPScore, d.human_score)[0],
                  "Kendall": kendalltau(d.CLIPScore, d.human_score)[0]})
corr = pd.DataFrame(corr_rows)
corr.to_csv(RES_DIR / "correlations_juge.csv", index=False)
corr.round(3)

In [ ]:
c = corr[corr["juge"] != "CLIPScore (référence)"]
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(c["juge"], c["Spearman"], color=[colors[j.split()[-1]] for j in c["juge"]])
ax.bar_label(bars, fmt="%.2f")
clip_s = corr[corr["juge"] == "CLIPScore (référence)"]["Spearman"].values[0]
ax.axhline(clip_s, ls="--", color="gray", label=f"CLIPScore ({clip_s:.2f})")
ax.set_ylabel("Corrélation de Spearman avec l'humain"); ax.set_ylim(0, 1)
ax.set_title("Accord VLM-juge / humain selon la quantification")
plt.xticks(rotation=20, ha="right"); ax.legend(); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(RES_DIR / "correlation_juge.png", dpi=150); plt.show()

## 8. Démo finale — Gradio

Charge une image, choisis le modèle, la **précision (16 / 8 / 4 bits)** et le mode :
- **Légende** : le VLM décrit l'image ;
- **Score** : le VLM note de 1 à 5 une légende (la tienne, ou à défaut celle qu'il génère).

Un seul modèle est gardé en mémoire à la fois. Le lien public `gradio.live` s'affiche sous la cellule.

In [ ]:
import gradio as gr

for _v in ("clip_model",):
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

_cur = {"key": None, "model": None, "proc": None}

def get_model(name, prec):
    if _cur["key"] != (name, prec):
        if _cur["model"] is not None:
            _cur["model"] = _cur["proc"] = None
            gc.collect(); torch.cuda.empty_cache()
        _cur["model"], _cur["proc"] = load_vlm(name, prec)
        _cur["key"] = (name, prec)
    return _cur["model"], _cur["proc"]

def demo_fn(image, name, prec, mode, user_caption):
    if image is None:
        return "Charge une image.", ""
    image = prepare_image(image)
    model, proc = get_model(name, prec)
    torch.cuda.synchronize(); t = time.time()
    if mode == "Légende":
        out = clean_caption(vlm_generate(model, proc, image, CAPTION_PROMPT))
    else:
        cap = user_caption.strip() or clean_caption(vlm_generate(model, proc, image, CAPTION_PROMPT))
        raw = vlm_generate(model, proc, image, JUDGE_PROMPT.format(caption=cap), max_new_tokens=5)
        s = parse_score(raw)
        out = f"Légende évaluée : « {cap} »\nScore : {s}/5" if not np.isnan(s) else f"Réponse non exploitable : {raw}"
    torch.cuda.synchronize()
    info = (f"{name} en {prec} — poids : {model.get_memory_footprint() / 1e9:.2f} Go — "
            f"temps : {time.time() - t:.2f} s")
    return out, info

with gr.Blocks(title="Projet 7 — VLMs légers") as demo:
    gr.Markdown("# Projet 7 — VLMs légers pour légender ou juger des images")
    with gr.Row():
        with gr.Column():
            img_in = gr.Image(type="pil", label="Image")
            name_in = gr.Dropdown(list(MODELS), value=list(MODELS)[0], label="Modèle")
            prec_in = gr.Radio(PRECISIONS, value="4bit", label="Niveau de quantification")
            mode_in = gr.Radio(["Légende", "Score"], value="Légende", label="Mode")
            cap_in = gr.Textbox(label="Légende à évaluer (mode Score, optionnel)")
            btn = gr.Button("Lancer", variant="primary")
        with gr.Column():
            out_txt = gr.Textbox(label="Résultat", lines=4)
            out_info = gr.Markdown()
    gr.Examples([[str(IMG_DIR / f"{i}.jpg")] for i in IDS[:4]], inputs=[img_in])
    btn.click(demo_fn, [img_in, name_in, prec_in, mode_in, cap_in], [out_txt, out_info])

demo.launch(share=True, debug=False)

## 9. Conclusion — à compléter avec tes résultats

Appuie-toi sur `results/resultats_finaux.csv`, `comparaison.png`, `compromis_memoire_qualite.png` et `correlation_juge.png`.

1. **Mémoire** : de combien le 8 bits et le 4 bits réduisent-ils la mémoire des poids et le pic GPU par rapport au 16 bits ?
2. **Vitesse** : le 8 bits (LLM.int8) est souvent **plus lent** que le 16 bits à cause de la décomposition des valeurs aberrantes ; le 4 bits NF4 est généralement entre les deux. Qu'observes-tu ?
3. **Qualité** : les pertes de CIDEr / CLIPScore sont-elles significatives ? Quel pourcentage de légendes reste identique au 16 bits ?
4. **Comparaison des deux modèles** : lequel légende le mieux ? Lequel supporte le mieux la quantification ?
5. **Option A** : quelle configuration recommandes-tu pour générer en masse des légendes d'entraînement LoRA (meilleur compromis qualité / coût) ?
6. **Option B** : le VLM juge-t-il comme un humain ? La quantification dégrade-t-elle la corrélation ? Fait-il mieux ou moins bien que le CLIPScore ?
7. **Limites** : sous-ensemble de N images, légendes VLM plus longues que Flickr8k (biais des métriques à base de n-grammes), un seul prompt testé, annotation humaine par une seule personne.